# Final Project: Random Forest Classifier

---

**Student Name:** Omar Al-Asfar

**Student ID:** alasfaro | 400369814

**CodaBench Username:** alasfaro

---

### AI Tool Usage Declaration

The original model featured simply a RandomForest model which was implementing by only referencing online resources cited below. ChatGPT was subsequently used to research ways to improve the model further, implementation of suggestions, and error debugging.

4 prompts were used, suggesting that the amount of CO2 emitted was 17.28g.

### Bag-of-Words Classifier

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from string import punctuation
from collections.abc import Iterable
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import scipy.sparse as sp

In [2]:
SEED = 42
np.random.seed(SEED)

In [3]:
nltk.download('stopwords')
stopwords_set = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\oalas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
def raw_text(df: pd.DataFrame):
    return (df['subject'].fillna('') + ' ' + df['body'].fillna('')).apply(lambda x : x.replace('\r\n', ' '))

In [5]:
def stop_stem(text: Iterable[str], stops: set[str], stems: PorterStemmer):
    return ' '.join([stems.stem(w) for w in text if w not in stops])

In [6]:
def preprocess(df: pd.DataFrame, stops: set[str], stems: PorterStemmer):
    df['text'] = [stop_stem(t.lower().translate(str.maketrans('', '', punctuation)).split(),
                            stopwords_set, stemmer) for t in raw_text(df)]

In [ ]:
# Custom added features using raw metadata
# These features are designed to capture common phishing signals suggested by cited sources and ChatGPT

def extract_features(df: pd.DataFrame) -> np.ndarray:
    feats = pd.DataFrame(index=df.index)
    body = df['body'].fillna('')
    subject = df['subject'].fillna('')
    urls_col = df['urls'].fillna('') if 'urls' in df.columns else pd.Series([''] * len(df), index=df.index)

    # Searches for URL signals
    feats['url_count'] = urls_col.apply(lambda x: len(str(x).split()) if x else 0)
    feats['has_url'] = (feats['url_count'] > 0).astype(int)

    # Subject signals
    feats['subject_len'] = subject.str.len()
    feats['subject_caps_ratio'] = subject.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    feats['subject_exclaim'] = subject.str.count('!')
    feats['subject_question'] = subject.str.count(r'\?')

    # Body signals
    feats['body_len'] = body.str.len()
    feats['body_exclaim'] = body.str.count('!')
    feats['body_caps_ratio'] = body.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    feats['body_dollar'] = body.str.count(r'\$')
    feats['body_urgent'] = body.str.lower().str.count(
        r'\b(urgent|verify|confirm|click|limited|offer|free|win|prize|account|update|suspend)\b'
    )

    # Sender domain signals which assume common free email providers are more likely to be used by phishers
    # On its own this is a weak signal but it can help when combined with the others
    sender = df['sender'].fillna('')
    feats['sender_free_domain'] = sender.str.lower().str.contains(
        r'gmail|yahoo|hotmail|outlook|aol', regex=True
    ).astype(int)

    return feats.values.astype(np.float32)

In [ ]:
train_csv = 'train.csv'
val_csv = 'val.csv'
test_csv = 'test.csv'

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

preprocess(train_df, stopwords_set, stemmer)
preprocess(val_df, stopwords_set, stemmer)
preprocess(test_df, stopwords_set, stemmer)

In [ ]:
# TF-IDF with bigrams
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=75000, min_df=2)
tfidf_train = vectorizer.fit_transform(train_df['text'])
hand_train = extract_features(train_df)
train_x = sp.hstack([tfidf_train, sp.csr_matrix(hand_train)])
train_y = train_df.label

In [ ]:
clf = RandomForestClassifier(
    n_estimators=500,
    max_features=500, # Added to give more signal per split
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
)
clf.fit(train_x, train_y)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,500
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
tfidf_val = vectorizer.transform(val_df['text'])
hand_val = extract_features(val_df)
val_x = sp.hstack([tfidf_val, sp.csr_matrix(hand_val)])
val_y = val_df.label

tfidf_test = vectorizer.transform(test_df['text'])
hand_test = extract_features(test_df)
test_x = sp.hstack([tfidf_test, sp.csr_matrix(hand_test)])

In [ ]:
from sklearn.metrics import f1_score

# Tune decision threshold on validation set
# Attempted 0.5 but was not optimal
val_probs = clf.predict_proba(val_x)[:, 1]
best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.30, 0.70, 0.01):
    preds = (val_probs >= t).astype(int)
    f = f1_score(val_y, preds)
    if f > best_f1:
        best_f1, best_thresh = f, t

val_preds = (val_probs >= best_thresh).astype(int)
val_acc = (val_preds == val_y).mean()
print(f"Best threshold: {best_thresh:.2f}")
print(f"Val F1:         {best_f1:.4f}")
print(f"Val Accuracy:   {val_acc:.4f}")

Best threshold: 0.44
Val F1:         0.9696
Val Accuracy:   0.9693


In [13]:
test_probs = clf.predict_proba(test_x)[:, 1]
test_y_pred = (test_probs >= best_thresh).astype(int)

In [14]:
submission_df = pd.DataFrame(test_y_pred, columns=['label'])
print(submission_df.head())
submission_df.to_csv('submission.csv', index=False)

   label
0      1
1      0
2      1
3      1
4      0


---
## Code Attribution

List any code snippets, tutorials, or resources you referenced below. Include URLs and a short description of what was adapted.

| Source | URL | What was adapted |
|--------|-----|------------------|
| HuggingFace Trainer tutorial | https://huggingface.co/docs/transformers/training | Fine-tuning loop scaffold |
| Spam Mail Detection with Machine Learning in Python | https://www.youtube.com/watch?v=nkPNQk4-3UE | Classifier technique |
| Learning to detect phishing emails | https://dl.acm.org/doi/10.1145/1242572.1242660 | Additional deined features |
| Detection of Phishing Attacks: A Machine Learning Approach | https://link.springer.com/chapter/10.1007/978-3-540-77465-5_19 | Sender domain signals |